In [27]:
import numpy as np
import pandas as pd
import time
import itertools
import contextlib
import io
from pathlib import Path

from raiselab.core import dvqe


# ============================================================
# EXPERIMENT SETTINGS
# ============================================================

N_VALUES = list(range(6, 16))          # n = 6, 7, ..
SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]             
OUTPUT_DIR = Path("dvqe_ablation_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# Same DVQE training settings for all variants
DVQE_SETTINGS = dict(
    depth=1,
    lr=0.05,
    max_iters=80,
    rel_tol=1e-5,
    num_shots=256,
    final_shots=2000,
    warm_start_population=6,
    warm_start_iters=8,
    warm_start_shots=64,
    energy_mode="cvar",
    cvar_alpha=0.2,
    verbose=False,
)

# Same distributed QPU configuration for all distributed runs

DISTRIBUTED_QPU_CONFIG = [4, 4, 4, 4, 4, 4]  

# Success tolerance against exact brute force optimum
SUCCESS_ABS_TOL = 1e-8

# Variants to compare
METHODS = [
    {
        "method": "monolithic_random",
        "mode": "monolithic",
        "init_type": 1,
        "qpu_qubit_config": None,
    },
    {
        "method": "distributed_random",
        "mode": "distributed",
        "init_type": 1,
        "qpu_qubit_config": DISTRIBUTED_QPU_CONFIG,
    },
    {
        "method": "distributed_BH",
        "mode": "distributed",
        "init_type": 2,
        "qpu_qubit_config": DISTRIBUTED_QPU_CONFIG,
    },
    {
        "method": "distributed_GWO",
        "mode": "distributed",
        "init_type": 3,
        "qpu_qubit_config": DISTRIBUTED_QPU_CONFIG,
    },
    {
        "method": "distributed_ABC",
        "mode": "distributed",
        "init_type": 4,
        "qpu_qubit_config": DISTRIBUTED_QPU_CONFIG,
    },
]


# ============================================================
# QUBO HELPERS
# ============================================================

def qubo_energy(Q, q, z):
    """
    Compute QUBO energy:
        C(z) = z^T Q z + q^T z
    """
    z = np.asarray(z, dtype=float)
    return float(z.T @ Q @ z + q.T @ z)


def scale_qubo(Q, q):
    """
    Scale QUBO coefficients for DVQE numerical stability.
    The minimizer is unchanged because the objective is divided
    by a positive constant.
    """
    scale = max(
        float(np.max(np.abs(Q))) if Q.size > 0 else 0.0,
        float(np.max(np.abs(q))) if q.size > 0 else 0.0,
        1.0,
    )
    return Q / scale, q / scale, scale


def generate_random_qubo(n, seed, density=0.65):
    """
    Generate a random QUBO instance that is not too easy and not too hard.

    The matrix is symmetric. Off-diagonal terms are randomly sparsified.
    Diagonal and linear terms are included so that the problem is not only
    pairwise interaction based.

    Since n <= 12 in this experiment, the global optimum is computed exactly
    by brute force and used as the reference.
    """
    rng = np.random.default_rng(seed)

    # Random symmetric quadratic part
    W = rng.normal(loc=0.0, scale=1.0, size=(n, n))
    mask = rng.random((n, n)) < density
    mask = np.triu(mask, 1)
    W = np.triu(W, 1) * mask
    Q = W + W.T

    # Add diagonal terms
    diag = rng.normal(loc=0.0, scale=0.75, size=n)
    Q[np.diag_indices(n)] = diag

    # Add linear terms
    q = rng.normal(loc=0.0, scale=0.75, size=n)

    # Symmetrize for numerical consistency
    Q = 0.5 * (Q + Q.T)

    return Q, q


def brute_force_qubo(Q, q):
    """
    Exact brute force QUBO solver for n <= 12.
    Returns the best binary vector and its objective value.
    """
    n = q.shape[0]
    best_z = None
    best_energy = np.inf

    for bits in itertools.product([0, 1], repeat=n):
        z = np.array(bits, dtype=int)
        e = qubo_energy(Q, q, z)
        if e < best_energy:
            best_energy = e
            best_z = z.copy()

    return best_z, best_energy


def hamming_distance(z1, z2):
    z1 = np.asarray(z1, dtype=int)
    z2 = np.asarray(z2, dtype=int)
    return int(np.sum(z1 != z2))


# ============================================================
# SINGLE DVQE RUN
# ============================================================

def run_one_dvqe_variant(Q, q, n, seed, method_cfg):
    """
    Run one DVQE variant on one QUBO instance.
    """
    Q_train, q_train, qubo_scale = scale_qubo(Q, q)

    run_seed = seed + 10000 * n + 100 * method_cfg["init_type"]

    start = time.time()

    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        z_best, final_circuit, hist = dvqe(
            mode=method_cfg["mode"],
            Q=Q_train,
            q_linear=q_train,
            init_type=method_cfg["init_type"],
            qpu_qubit_config=method_cfg["qpu_qubit_config"],
            seed=run_seed,
            **DVQE_SETTINGS,
        )

    runtime = time.time() - start

    z_best = np.asarray(z_best, dtype=int)

    # Evaluate returned solution using the original unscaled QUBO.
    energy_original = qubo_energy(Q, q, z_best)

    return {
        "z_best": z_best,
        "energy_original": energy_original,
        "runtime_sec": runtime,
        "hist_size": len(hist) if hist is not None else None,
        "qubo_scale": qubo_scale,
    }


# ============================================================
# MAIN EXPERIMENT RUNNER
# ============================================================

def run_dvqe_ablation_experiment():
    """
    Run DVQE ablation study.

    Compared methods:
        1) monolithic DVQE with random initialization
        2) distributed DVQE with random initialization
        3) distributed DVQE with BH warm start
        4) distributed DVQE with GWO warm start
        5) distributed DVQE with ABC warm start

    For every n and seed, the same QUBO is solved by all variants.
    The exact optimum is computed by brute force and used as reference.
    """

    detailed_rows = []

    total_runs = len(N_VALUES) * len(SEEDS) * len(METHODS)
    completed = 0

    print("=" * 90)
    print("DVQE ABLATION STUDY")
    print("=" * 90)
    print(f"Problem sizes: {N_VALUES}")
    print(f"Seeds: {SEEDS}")
    print(f"Total DVQE runs: {total_runs}")
    print(f"Distributed QPU config: {DISTRIBUTED_QPU_CONFIG}")
    print("=" * 90)

    for n in N_VALUES:
        for seed in SEEDS:

            qubo_seed = 1000 * n + seed
            Q, q = generate_random_qubo(n=n, seed=qubo_seed)

            z_opt, f_opt = brute_force_qubo(Q, q)

            print(f"\nQUBO instance | n={n:2d} | seed={seed:2d} | exact optimum={f_opt:.8f}")

            for method_cfg in METHODS:
                method_name = method_cfg["method"]

                try:
                    result = run_one_dvqe_variant(
                        Q=Q,
                        q=q,
                        n=n,
                        seed=seed,
                        method_cfg=method_cfg,
                    )

                    z_best = result["z_best"]
                    f_dvqe = result["energy_original"]

                    abs_error = float(f_dvqe - f_opt)
                    rel_error = float(abs_error / max(1.0, abs(f_opt)))
                    success = bool(abs_error <= SUCCESS_ABS_TOL)
                    ham = hamming_distance(z_best, z_opt)

                    row = {
                        "n": n,
                        "seed": seed,
                        "qubo_seed": qubo_seed,
                        "method": method_name,
                        "mode": method_cfg["mode"],
                        "init_type": method_cfg["init_type"],
                        "exact_objective": f_opt,
                        "dvqe_objective": f_dvqe,
                        "abs_error": abs_error,
                        "rel_error": rel_error,
                        "success": success,
                        "hamming_to_exact": ham,
                        "runtime_sec": result["runtime_sec"],
                        "hist_size": result["hist_size"],
                        "qubo_scale": result["qubo_scale"],
                        "z_exact": "".join(map(str, z_opt.tolist())),
                        "z_dvqe": "".join(map(str, z_best.tolist())),
                        "status": "ok",
                    }

                    print(
                        f"  {method_name:22s} | "
                        f"obj={f_dvqe: .8f} | "
                        f"err={abs_error: .3e} | "
                        f"success={success} | "
                        f"time={result['runtime_sec']: .2f}s"
                    )

                except Exception as e:
                    row = {
                        "n": n,
                        "seed": seed,
                        "qubo_seed": qubo_seed,
                        "method": method_name,
                        "mode": method_cfg["mode"],
                        "init_type": method_cfg["init_type"],
                        "exact_objective": f_opt,
                        "dvqe_objective": np.nan,
                        "abs_error": np.nan,
                        "rel_error": np.nan,
                        "success": False,
                        "hamming_to_exact": np.nan,
                        "runtime_sec": np.nan,
                        "hist_size": np.nan,
                        "qubo_scale": np.nan,
                        "z_exact": "".join(map(str, z_opt.tolist())),
                        "z_dvqe": "",
                        "status": f"failed: {str(e)}",
                    }

                    print(f"  {method_name:22s} | FAILED | {str(e)}")

                detailed_rows.append(row)
                completed += 1

    detailed_df = pd.DataFrame(detailed_rows)

    # ------------------------------------------------------------
    # Summary by method
    # ------------------------------------------------------------

    summary_by_method = (
        detailed_df
        .groupby("method", dropna=False)
        .agg(
            runs=("success", "count"),
            successes=("success", "sum"),
            success_rate=("success", "mean"),
            mean_abs_error=("abs_error", "mean"),
            max_abs_error=("abs_error", "max"),
            median_abs_error=("abs_error", "median"),
            mean_rel_error=("rel_error", "mean"),
            max_rel_error=("rel_error", "max"),
            mean_hamming=("hamming_to_exact", "mean"),
            max_hamming=("hamming_to_exact", "max"),
            mean_runtime_sec=("runtime_sec", "mean"),
            total_runtime_sec=("runtime_sec", "sum"),
        )
        .reset_index()
    )

    # ------------------------------------------------------------
    # Summary by problem size and method
    # ------------------------------------------------------------

    summary_by_n_method = (
        detailed_df
        .groupby(["n", "method"], dropna=False)
        .agg(
            runs=("success", "count"),
            successes=("success", "sum"),
            success_rate=("success", "mean"),
            mean_abs_error=("abs_error", "mean"),
            max_abs_error=("abs_error", "max"),
            mean_runtime_sec=("runtime_sec", "mean"),
        )
        .reset_index()
    )

    # ------------------------------------------------------------
    # Monolithic random vs distributed random comparison
    # ------------------------------------------------------------

    random_compare = compare_monolithic_distributed_random(detailed_df)

    # ------------------------------------------------------------
    # Save files
    # ------------------------------------------------------------

    detailed_path = OUTPUT_DIR / "dvqe_ablation_detailed.csv"
    summary_method_path = OUTPUT_DIR / "dvqe_ablation_summary_by_method.csv"
    summary_n_method_path = OUTPUT_DIR / "dvqe_ablation_summary_by_n_method.csv"
    random_compare_path = OUTPUT_DIR / "dvqe_monolithic_vs_distributed_random.csv"

    detailed_df.to_csv(detailed_path, index=False)
    summary_by_method.to_csv(summary_method_path, index=False)
    summary_by_n_method.to_csv(summary_n_method_path, index=False)
    random_compare.to_csv(random_compare_path, index=False)

    print("\n" + "=" * 90)
    print("SUMMARY BY METHOD")
    print("=" * 90)
    print(summary_by_method.to_string(index=False))

    print("\n" + "=" * 90)
    print("MONOLITHIC RANDOM VS DISTRIBUTED RANDOM")
    print("=" * 90)
    print(random_compare.to_string(index=False))

    print("\n" + "=" * 90)
    print("FILES SAVED")
    print("=" * 90)
    print(detailed_path)
    print(summary_method_path)
    print(summary_n_method_path)
    print(random_compare_path)

    return detailed_df, summary_by_method, summary_by_n_method, random_compare


# ============================================================
# COMPARISON HELPERS
# ============================================================

def compare_monolithic_distributed_random(detailed_df):
    """
    Compare monolithic random and distributed random on the same QUBOs.

    This is used to support the statement that distributed DVQE returns
    comparable results to monolithic DVQE under the same training settings.
    """
    mono = detailed_df[detailed_df["method"] == "monolithic_random"].copy()
    dist = detailed_df[detailed_df["method"] == "distributed_random"].copy()

    keep_cols = [
        "n",
        "seed",
        "exact_objective",
        "dvqe_objective",
        "abs_error",
        "success",
        "runtime_sec",
        "z_dvqe",
    ]

    mono = mono[keep_cols].rename(columns={
        "dvqe_objective": "monolithic_objective",
        "abs_error": "monolithic_abs_error",
        "success": "monolithic_success",
        "runtime_sec": "monolithic_runtime_sec",
        "z_dvqe": "monolithic_z",
    })

    dist = dist[keep_cols].rename(columns={
        "dvqe_objective": "distributed_objective",
        "abs_error": "distributed_abs_error",
        "success": "distributed_success",
        "runtime_sec": "distributed_runtime_sec",
        "z_dvqe": "distributed_z",
    })

    merged = pd.merge(
        mono,
        dist,
        on=["n", "seed", "exact_objective"],
        how="inner",
    )

    merged["objective_difference_dist_minus_mono"] = (
        merged["distributed_objective"] - merged["monolithic_objective"]
    )

    merged["same_objective"] = np.isclose(
        merged["distributed_objective"],
        merged["monolithic_objective"],
        atol=SUCCESS_ABS_TOL,
        rtol=0.0,
    )

    merged["same_solution_string"] = (
        merged["distributed_z"] == merged["monolithic_z"]
    )

    return merged


# ============================================================
# OPTIONAL LATEX TABLE GENERATOR
# ============================================================

def make_latex_tables(summary_by_method, summary_by_n_method, random_compare):
    """
    Create LaTeX tables for the paper.
    """

    table_method = summary_by_method.copy()
    table_method["success_rate"] = 100.0 * table_method["success_rate"]

    table_method = table_method[
        [
            "method",
            "runs",
            "successes",
            "success_rate",
            "mean_abs_error",
            "max_abs_error",
            "mean_runtime_sec",
        ]
    ]

    latex_method = table_method.to_latex(
        index=False,
        float_format=lambda x: f"{x:.4e}" if abs(x) < 1e-2 or abs(x) > 1e3 else f"{x:.4f}",
        caption="DVQE ablation summary over all QUBO instances.",
        label="tab:dvqe_ablation_summary",
    )

    table_n = summary_by_n_method.copy()
    table_n["success_rate"] = 100.0 * table_n["success_rate"]

    latex_n = table_n.to_latex(
        index=False,
        float_format=lambda x: f"{x:.4e}" if abs(x) < 1e-2 or abs(x) > 1e3 else f"{x:.4f}",
        caption="DVQE ablation summary by QUBO size.",
        label="tab:dvqe_ablation_by_size",
    )

    random_summary = pd.DataFrame({
        "metric": [
            "number of paired instances",
            "same objective count",
            "same solution string count",
            "mean objective difference",
            "max absolute objective difference",
            "monolithic success count",
            "distributed success count",
            "mean monolithic runtime",
            "mean distributed runtime",
        ],
        "value": [
            len(random_compare),
            int(random_compare["same_objective"].sum()),
            int(random_compare["same_solution_string"].sum()),
            float(random_compare["objective_difference_dist_minus_mono"].mean()),
            float(np.abs(random_compare["objective_difference_dist_minus_mono"]).max()),
            int(random_compare["monolithic_success"].sum()),
            int(random_compare["distributed_success"].sum()),
            float(random_compare["monolithic_runtime_sec"].mean()),
            float(random_compare["distributed_runtime_sec"].mean()),
        ],
    })

    latex_random = random_summary.to_latex(
        index=False,
        float_format=lambda x: f"{x:.4e}" if abs(x) < 1e-2 or abs(x) > 1e3 else f"{x:.4f}",
        caption="Comparison between monolithic random DVQE and distributed random DVQE.",
        label="tab:dvqe_mono_dist_random",
    )

    with open(OUTPUT_DIR / "table_dvqe_ablation_summary.tex", "w") as f:
        f.write(latex_method)

    with open(OUTPUT_DIR / "table_dvqe_ablation_by_size.tex", "w") as f:
        f.write(latex_n)

    with open(OUTPUT_DIR / "table_dvqe_mono_dist_random.tex", "w") as f:
        f.write(latex_random)

    return latex_method, latex_n, latex_random




In [28]:
detailed_df, summary_by_method, summary_by_n_method, random_compare = run_dvqe_ablation_experiment()

latex_method, latex_n, latex_random = make_latex_tables(
        summary_by_method,
        summary_by_n_method,
        random_compare,
    )

DVQE ABLATION STUDY
Problem sizes: [6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Seeds: [1, 2, 3, 4, 5, 6, 7, 8]
Total DVQE runs: 400
Distributed QPU config: [4, 4, 4, 4, 4, 4]

QUBO instance | n= 6 | seed= 1 | exact optimum=-6.88938850
  monolithic_random      | obj=-6.88938850 | err= 0.000e+00 | success=True | time= 0.41s
  distributed_random     | obj=-6.88938850 | err= 0.000e+00 | success=True | time= 0.39s
  distributed_BH         | obj=-6.88938850 | err= 0.000e+00 | success=True | time= 0.48s
  distributed_GWO        | obj=-6.88938850 | err= 0.000e+00 | success=True | time= 0.53s
  distributed_ABC        | obj=-6.88938850 | err= 0.000e+00 | success=True | time= 0.56s

QUBO instance | n= 6 | seed= 2 | exact optimum=-3.29754134
  monolithic_random      | obj=-3.29754134 | err= 0.000e+00 | success=True | time= 0.48s
  distributed_random     | obj=-3.29754134 | err= 0.000e+00 | success=True | time= 0.46s
  distributed_BH         | obj=-3.29754134 | err= 0.000e+00 | success=True | time= 0.62s
